In [21]:
"""
Financial Feature Engineering Script
======================================
Creates advanced financial metrics and ratios from balance sheet and income statement data.
Handles missing values intelligently and normalizes data for machine learning.

Features Created:
- Liquidity Ratios
- Profitability Ratios
- Leverage/Solvency Ratios
- Efficiency Ratios
- Growth Metrics
- Quality of Earnings
- Cash Flow Proxies
- Operational Efficiency
"""

import pandas as pd
import numpy as np
from typing import Tuple, Dict
import warnings
warnings.filterwarnings('ignore')


class FinancialFeatureEngineer:
    """
    Create financial features with intelligent handling of missing values.
    Each feature is built with availability checking.
    """
    
    def __init__(self, df: pd.DataFrame, min_data_availability: float = 0.3):
        """
        Parameters:
        -----------
        df : pd.DataFrame
            DataFrame with financial data
        min_data_availability : float
            Minimum % of non-null values required to create a feature (0.0-1.0)
        """
        self.df = df.copy()
        self.min_availability = min_data_availability
        self.created_features = {}
        self.feature_availability = {}
        
    def _check_availability(self, columns: list) -> float:
        """Calculate data availability for a list of columns"""
        if not all(col in self.df.columns for col in columns):
            return 0.0
        return self.df[columns].notna().all(axis=1).sum() / len(self.df)
    
    def _safe_divide(self, numerator, denominator, fill_value: float = np.nan) -> pd.Series:
        """Safely divide avoiding division by zero"""
        result = numerator.copy().astype(float)
        denominator = denominator.astype(float)
        mask = (denominator != 0) & (denominator.notna()) & (numerator.notna())
        result[~mask] = fill_value
        result[mask] = numerator[mask] / denominator[mask]
        return result
    
    # ==================== LIQUIDITY RATIOS ====================
    
    def current_ratio(self) -> pd.Series:
        """Current Assets / Current Liabilities - Ability to pay short-term obligations"""
        cols = ['totalCurrentAssets', 'totalCurrentLiabilities']
        avail = self._check_availability(cols)
        if avail < self.min_availability:
            return pd.Series(np.nan, index=self.df.index)
        
        ratio = self._safe_divide(self.df['totalCurrentAssets'], 
                                   self.df['totalCurrentLiabilities'])
        self.feature_availability['current_ratio'] = avail
        return ratio
    
    def quick_ratio(self) -> pd.Series:
        """(Current Assets - Inventory) / Current Liabilities - More conservative liquidity"""
        cols = ['totalCurrentAssets', 'inventory', 'totalCurrentLiabilities']
        avail = self._check_availability(cols)
        if avail < self.min_availability:
            return pd.Series(np.nan, index=self.df.index)
        
        quick_assets = self.df['totalCurrentAssets'] - self.df['inventory'].fillna(0)
        ratio = self._safe_divide(quick_assets, self.df['totalCurrentLiabilities'])
        self.feature_availability['quick_ratio'] = avail
        return ratio
    
    def cash_ratio(self) -> pd.Series:
        """(Cash + Investments) / Current Liabilities - Most conservative liquidity"""
        cols = ['cashAndCashEquivalentsAtCarryingValue', 'shortTermInvestments', 
                'totalCurrentLiabilities']
        avail = self._check_availability(cols)
        if avail < self.min_availability:
            return pd.Series(np.nan, index=self.df.index)
        
        cash = (self.df['cashAndCashEquivalentsAtCarryingValue'].fillna(0) + 
                self.df['shortTermInvestments'].fillna(0))
        ratio = self._safe_divide(cash, self.df['totalCurrentLiabilities'])
        self.feature_availability['cash_ratio'] = avail
        return ratio
    
    def working_capital_ratio(self) -> pd.Series:
        """Working Capital / Total Assets - Capital efficiency"""
        cols = ['totalCurrentAssets', 'totalCurrentLiabilities', 'totalAssets']
        avail = self._check_availability(cols)
        if avail < self.min_availability:
            return pd.Series(np.nan, index=self.df.index)
        
        working_capital = self.df['totalCurrentAssets'] - self.df['totalCurrentLiabilities']
        ratio = self._safe_divide(working_capital, self.df['totalAssets'])
        self.feature_availability['working_capital_ratio'] = avail
        return ratio
    
    # ==================== PROFITABILITY RATIOS ====================
    
    def gross_profit_margin(self) -> pd.Series:
        """Gross Profit / Total Revenue - Production efficiency"""
        cols = ['grossProfit', 'totalRevenue']
        avail = self._check_availability(cols)
        if avail < self.min_availability:
            return pd.Series(np.nan, index=self.df.index)
        
        margin = self._safe_divide(self.df['grossProfit'], self.df['totalRevenue'])
        self.feature_availability['gross_profit_margin'] = avail
        return margin
    
    def operating_profit_margin(self) -> pd.Series:
        """Operating Income / Total Revenue - Core operational efficiency"""
        cols = ['operatingIncome', 'totalRevenue']
        avail = self._check_availability(cols)
        if avail < self.min_availability:
            return pd.Series(np.nan, index=self.df.index)
        
        margin = self._safe_divide(self.df['operatingIncome'], self.df['totalRevenue'])
        self.feature_availability['operating_profit_margin'] = avail
        return margin
    
    def net_profit_margin(self) -> pd.Series:
        """Net Income / Total Revenue - Bottom line profitability"""
        cols = ['netIncome', 'totalRevenue']
        avail = self._check_availability(cols)
        if avail < self.min_availability:
            return pd.Series(np.nan, index=self.df.index)
        
        margin = self._safe_divide(self.df['netIncome'], self.df['totalRevenue'])
        self.feature_availability['net_profit_margin'] = avail
        return margin
    
    def ebitda_margin(self) -> pd.Series:
        """EBITDA / Total Revenue - Operating performance before financing"""
        cols = ['ebitda', 'totalRevenue']
        avail = self._check_availability(cols)
        if avail < self.min_availability:
            return pd.Series(np.nan, index=self.df.index)
        
        margin = self._safe_divide(self.df['ebitda'], self.df['totalRevenue'])
        self.feature_availability['ebitda_margin'] = avail
        return margin
    
    def roa(self) -> pd.Series:
        """Return on Assets: Net Income / Average Total Assets"""
        cols = ['netIncome', 'totalAssets']
        avail = self._check_availability(cols)
        if avail < self.min_availability:
            return pd.Series(np.nan, index=self.df.index)
        
        roa = self._safe_divide(self.df['netIncome'], self.df['totalAssets'])
        self.feature_availability['roa'] = avail
        return roa
    
    def roe(self) -> pd.Series:
        """Return on Equity: Net Income / Total Shareholder Equity"""
        cols = ['netIncome', 'totalShareholderEquity']
        avail = self._check_availability(cols)
        if avail < self.min_availability:
            return pd.Series(np.nan, index=self.df.index)
        
        roe = self._safe_divide(self.df['netIncome'], self.df['totalShareholderEquity'])
        self.feature_availability['roe'] = avail
        return roe
    
    # ==================== LEVERAGE / SOLVENCY RATIOS ====================
    
    def debt_to_equity(self) -> pd.Series:
        """Total Liabilities / Total Equity - Financial leverage"""
        cols = ['totalLiabilities', 'totalShareholderEquity']
        avail = self._check_availability(cols)
        if avail < self.min_availability:
            return pd.Series(np.nan, index=self.df.index)
        
        ratio = self._safe_divide(self.df['totalLiabilities'], self.df['totalShareholderEquity'])
        self.feature_availability['debt_to_equity'] = avail
        return ratio
    
    def debt_to_assets(self) -> pd.Series:
        """Total Liabilities / Total Assets - Financial risk"""
        cols = ['totalLiabilities', 'totalAssets']
        avail = self._check_availability(cols)
        if avail < self.min_availability:
            return pd.Series(np.nan, index=self.df.index)
        
        ratio = self._safe_divide(self.df['totalLiabilities'], self.df['totalAssets'])
        self.feature_availability['debt_to_assets'] = avail
        return ratio
    
    def equity_ratio(self) -> pd.Series:
        """Total Equity / Total Assets - Asset financing from equity"""
        cols = ['totalShareholderEquity', 'totalAssets']
        avail = self._check_availability(cols)
        if avail < self.min_availability:
            return pd.Series(np.nan, index=self.df.index)
        
        ratio = self._safe_divide(self.df['totalShareholderEquity'], self.df['totalAssets'])
        self.feature_availability['equity_ratio'] = avail
        return ratio
    
    def interest_coverage(self) -> pd.Series:
        """EBIT / Interest Expense - Ability to pay interest"""
        cols = ['ebit', 'interestExpense']
        avail = self._check_availability(cols)
        if avail < self.min_availability:
            return pd.Series(np.nan, index=self.df.index)
        
        coverage = self._safe_divide(self.df['ebit'], self.df['interestExpense'])
        self.feature_availability['interest_coverage'] = avail
        return coverage
    
    def debt_service_coverage(self) -> pd.Series:
        """EBITDA / (Interest + Current Debt) - Ability to service debt"""
        cols = ['ebitda', 'interestExpense', 'currentDebt']
        avail = self._check_availability(cols)
        if avail < self.min_availability:
            return pd.Series(np.nan, index=self.df.index)
        
        debt_service = self.df['interestExpense'].fillna(0) + self.df['currentDebt'].fillna(0)
        coverage = self._safe_divide(self.df['ebitda'], debt_service)
        self.feature_availability['debt_service_coverage'] = avail
        return coverage
    
    # ==================== EFFICIENCY RATIOS ====================
    
    def asset_turnover(self) -> pd.Series:
        """Total Revenue / Total Assets - Asset utilization"""
        cols = ['totalRevenue', 'totalAssets']
        avail = self._check_availability(cols)
        if avail < self.min_availability:
            return pd.Series(np.nan, index=self.df.index)
        
        turnover = self._safe_divide(self.df['totalRevenue'], self.df['totalAssets'])
        self.feature_availability['asset_turnover'] = avail
        return turnover
    
    def receivables_turnover(self) -> pd.Series:
        """Total Revenue / Current Net Receivables - Collection efficiency"""
        cols = ['totalRevenue', 'currentNetReceivables']
        avail = self._check_availability(cols)
        if avail < self.min_availability:
            return pd.Series(np.nan, index=self.df.index)
        
        turnover = self._safe_divide(self.df['totalRevenue'], self.df['currentNetReceivables'])
        self.feature_availability['receivables_turnover'] = avail
        return turnover
    
    def inventory_turnover(self) -> pd.Series:
        """Cost of Revenue / Inventory - Inventory management"""
        cols = ['costOfRevenue', 'inventory']
        avail = self._check_availability(cols)
        if avail < self.min_availability:
            return pd.Series(np.nan, index=self.df.index)
        
        turnover = self._safe_divide(self.df['costOfRevenue'], self.df['inventory'])
        self.feature_availability['inventory_turnover'] = avail
        return turnover
    
    def days_inventory_outstanding(self) -> pd.Series:
        """365 / Inventory Turnover - Days inventory held"""
        inv_turnover = self.inventory_turnover()
        dio = 365.0 / inv_turnover
        self.feature_availability['days_inventory_outstanding'] = self.feature_availability.get('inventory_turnover', 0)
        return dio
    
    def days_sales_outstanding(self) -> pd.Series:
        """365 / Receivables Turnover - Days to collect payment"""
        rec_turnover = self.receivables_turnover()
        dso = 365.0 / rec_turnover
        self.feature_availability['days_sales_outstanding'] = self.feature_availability.get('receivables_turnover', 0)
        return dso
    
    # ==================== OPERATIONAL EFFICIENCY ====================
    
    def operating_expense_ratio(self) -> pd.Series:
        """Operating Expenses / Total Revenue - Cost control"""
        cols = ['operatingExpenses', 'totalRevenue']
        avail = self._check_availability(cols)
        if avail < self.min_availability:
            return pd.Series(np.nan, index=self.df.index)
        
        ratio = self._safe_divide(self.df['operatingExpenses'], self.df['totalRevenue'])
        self.feature_availability['operating_expense_ratio'] = avail
        return ratio
    
    def sga_ratio(self) -> pd.Series:
        """SG&A / Total Revenue - Administrative burden"""
        cols = ['sellingGeneralAndAdministrative', 'totalRevenue']
        avail = self._check_availability(cols)
        if avail < self.min_availability:
            return pd.Series(np.nan, index=self.df.index)
        
        ratio = self._safe_divide(self.df['sellingGeneralAndAdministrative'], self.df['totalRevenue'])
        self.feature_availability['sga_ratio'] = avail
        return ratio
    
    def rd_intensity(self) -> pd.Series:
        """R&D / Total Revenue - Innovation investment"""
        cols = ['researchAndDevelopment', 'totalRevenue']
        avail = self._check_availability(cols)
        if avail < self.min_availability:
            return pd.Series(np.nan, index=self.df.index)
        
        ratio = self._safe_divide(self.df['researchAndDevelopment'], self.df['totalRevenue'])
        self.feature_availability['rd_intensity'] = avail
        return ratio
    
    # ==================== QUALITY OF EARNINGS ====================
    
    def quality_of_earnings(self) -> pd.Series:
        """Operating Cash Flow / Net Income - Earnings sustainability
        Using EBITDA as proxy for operating cash flow"""
        cols = ['ebitda', 'netIncome']
        avail = self._check_availability(cols)
        if avail < self.min_availability:
            return pd.Series(np.nan, index=self.df.index)
        
        quality = self._safe_divide(self.df['ebitda'], self.df['netIncome'])
        self.feature_availability['quality_of_earnings'] = avail
        return quality
    
    def depreciation_to_ppe(self) -> pd.Series:
        """Depreciation / PPE - Asset aging indicator"""
        cols = ['depreciationAndAmortization', 'propertyPlantEquipment']
        avail = self._check_availability(cols)
        if avail < self.min_availability:
            return pd.Series(np.nan, index=self.df.index)
        
        ratio = self._safe_divide(self.df['depreciationAndAmortization'], 
                                  self.df['propertyPlantEquipment'])
        self.feature_availability['depreciation_to_ppe'] = avail
        return ratio
    
    # ==================== BALANCE SHEET HEALTH ====================
    
    def intangible_asset_ratio(self) -> pd.Series:
        """Intangible Assets / Total Assets - Asset quality"""
        cols = ['intangibleAssets', 'totalAssets']
        avail = self._check_availability(cols)
        if avail < self.min_availability:
            return pd.Series(np.nan, index=self.df.index)
        
        ratio = self._safe_divide(self.df['intangibleAssets'], self.df['totalAssets'])
        self.feature_availability['intangible_asset_ratio'] = avail
        return ratio
    
    def goodwill_ratio(self) -> pd.Series:
        """Goodwill / Total Assets - Acquisition impact"""
        cols = ['goodwill', 'totalAssets']
        avail = self._check_availability(cols)
        if avail < self.min_availability:
            return pd.Series(np.nan, index=self.df.index)
        
        ratio = self._safe_divide(self.df['goodwill'], self.df['totalAssets'])
        self.feature_availability['goodwill_ratio'] = avail
        return ratio
    
    def tangible_asset_ratio(self) -> pd.Series:
        """(Total Assets - Intangibles) / Total Assets - Tangible asset percentage"""
        cols = ['totalAssets', 'intangibleAssets']
        avail = self._check_availability(cols)
        if avail < self.min_availability:
            return pd.Series(np.nan, index=self.df.index)
        
        tangible = self.df['totalAssets'] - self.df['intangibleAssets'].fillna(0)
        ratio = self._safe_divide(tangible, self.df['totalAssets'])
        self.feature_availability['tangible_asset_ratio'] = avail
        return ratio
    
    # ==================== INCOME STATEMENT COMPOSITION ====================
    
    def cost_of_revenue_ratio(self) -> pd.Series:
        """Cost of Revenue / Total Revenue - Production costs"""
        cols = ['costOfRevenue', 'totalRevenue']
        avail = self._check_availability(cols)
        if avail < self.min_availability:
            return pd.Series(np.nan, index=self.df.index)
        
        ratio = self._safe_divide(self.df['costOfRevenue'], self.df['totalRevenue'])
        self.feature_availability['cost_of_revenue_ratio'] = avail
        return ratio
    
    def tax_rate(self) -> pd.Series:
        """Income Tax Expense / Income Before Tax - Effective tax rate"""
        cols = ['incomeTaxExpense', 'incomeBeforeTax']
        avail = self._check_availability(cols)
        if avail < self.min_availability:
            return pd.Series(np.nan, index=self.df.index)
        
        rate = self._safe_divide(self.df['incomeTaxExpense'], self.df['incomeBeforeTax'])
        self.feature_availability['tax_rate'] = avail
        return rate
    
    def ebit_to_revenue(self) -> pd.Series:
        """EBIT / Total Revenue - Core operational profitability"""
        cols = ['ebit', 'totalRevenue']
        avail = self._check_availability(cols)
        if avail < self.min_availability:
            return pd.Series(np.nan, index=self.df.index)
        
        ratio = self._safe_divide(self.df['ebit'], self.df['totalRevenue'])
        self.feature_availability['ebit_to_revenue'] = avail
        return ratio
    
    # ==================== CAPITAL STRUCTURE ====================
    
    def long_term_debt_ratio(self) -> pd.Series:
        """Long-term Debt / Total Assets - Long-term leverage"""
        cols = ['longTermDebt', 'totalAssets']
        avail = self._check_availability(cols)
        if avail < self.min_availability:
            return pd.Series(np.nan, index=self.df.index)
        
        ratio = self._safe_divide(self.df['longTermDebt'], self.df['totalAssets'])
        self.feature_availability['long_term_debt_ratio'] = avail
        return ratio
    
    def short_term_debt_ratio(self) -> pd.Series:
        """Short-term Debt / Total Assets - Short-term leverage"""
        cols = ['shortTermDebt', 'totalAssets']
        avail = self._check_availability(cols)
        if avail < self.min_availability:
            return pd.Series(np.nan, index=self.df.index)
        
        ratio = self._safe_divide(self.df['shortTermDebt'], self.df['totalAssets'])
        self.feature_availability['short_term_debt_ratio'] = avail
        return ratio
    
    def debt_composition(self) -> pd.Series:
        """Short-term Debt / Total Debt - Debt maturity profile"""
        cols = ['shortTermDebt', 'totalLiabilities']
        avail = self._check_availability(cols)
        if avail < self.min_availability:
            return pd.Series(np.nan, index=self.df.index)
        
        ratio = self._safe_divide(self.df['shortTermDebt'], self.df['totalLiabilities'])
        self.feature_availability['debt_composition'] = avail
        return ratio

    def other_raw_features(self) -> pd.Series:
        """Short-term Debt / Total Debt - Debt maturity profile"""
        cols = ['shortTermDebt', 'totalLiabilities']
        avail = self._check_availability(cols)
        if avail < self.min_availability:
            return pd.Series(np.nan, index=self.df.index)
        
        ratio = self._safe_divide(self.df['shortTermDebt'], self.df['totalLiabilities'])
        self.feature_availability['debt_composition'] = avail
        return ratio
    # ==================== GENERATE ALL FEATURES ====================
    
    def create_all_features(self) -> pd.DataFrame:
        """Create all financial features and return as DataFrame"""
        
        features_dict = {
            # Liquidity
            'current_ratio': self.current_ratio(),
            'quick_ratio': self.quick_ratio(),
            'cash_ratio': self.cash_ratio(),
            'working_capital_ratio': self.working_capital_ratio(),
            
            # Profitability
            'gross_profit_margin': self.gross_profit_margin(),
            'operating_profit_margin': self.operating_profit_margin(),
            'net_profit_margin': self.net_profit_margin(),
            'ebitda_margin': self.ebitda_margin(),
            'roa': self.roa(),
            'roe': self.roe(),
            
            # Leverage
            'debt_to_equity': self.debt_to_equity(),
            'debt_to_assets': self.debt_to_assets(),
            'equity_ratio': self.equity_ratio(),
            'interest_coverage': self.interest_coverage(),
            'debt_service_coverage': self.debt_service_coverage(),
            
            # Efficiency
            'asset_turnover': self.asset_turnover(),
            'receivables_turnover': self.receivables_turnover(),
            'inventory_turnover': self.inventory_turnover(),
            'days_inventory_outstanding': self.days_inventory_outstanding(),
            'days_sales_outstanding': self.days_sales_outstanding(),
            
            # Operations
            'operating_expense_ratio': self.operating_expense_ratio(),
            'sga_ratio': self.sga_ratio(),
            'rd_intensity': self.rd_intensity(),
            
            # Quality & Composition
            'quality_of_earnings': self.quality_of_earnings(),
            'depreciation_to_ppe': self.depreciation_to_ppe(),
            'cost_of_revenue_ratio': self.cost_of_revenue_ratio(),
            'tax_rate': self.tax_rate(),
            'ebit_to_revenue': self.ebit_to_revenue(),
            
            # Balance Sheet
            'intangible_asset_ratio': self.intangible_asset_ratio(),
            'goodwill_ratio': self.goodwill_ratio(),
            'tangible_asset_ratio': self.tangible_asset_ratio(),
            
            # Capital Structure
            'long_term_debt_ratio': self.long_term_debt_ratio(),
            'short_term_debt_ratio': self.short_term_debt_ratio(),
            'debt_composition': self.debt_composition(),
            'log_assets': np.log(self.df["totalAssets"]),
            'log_revenue': np.log1p(df["totalRevenue"]),
            'log_net_income': np.log1p(df["netIncome"]),
            'share_growth': self.df.groupby("ticker")["commonStockSharesOutstanding"].pct_change()
        }
        
        features_df = pd.DataFrame(features_dict)
        
        # Add original ticker and date columns if available
        if 'ticker' in self.df.columns:
            features_df.insert(0, 'ticker', self.df['ticker'])
        if 'fiscalDateEnding' in self.df.columns:
            features_df.insert(1, 'fiscalDateEnding', self.df['fiscalDateEnding'])
        
        self.created_features = features_df
        return features_df
    
    def get_feature_availability_report(self) -> pd.DataFrame:
        """Return report of feature availability (% of non-null rows)"""
        report = pd.DataFrame({
            'feature': list(self.feature_availability.keys()),
            'availability_%': [v * 100 for v in self.feature_availability.values()]
        }).sort_values('availability_%', ascending=False)
        
        return report
    
    def get_high_availability_features(self, threshold: float = 0.5) -> list:
        """Get features with availability above threshold"""
        return [f for f, avail in self.feature_availability.items() 
                if avail >= threshold]


# ==================== USAGE EXAMPLE ====================

if __name__ == "__main__":
    # Example: Load your data
    # df = pd.read_csv('your_financial_data.csv')
    
    # Create feature engineer
    # engineer = FinancialFeatureEngineer(df, min_data_availability=0.3)
    
    # Generate all features
    # features_df = engineer.create_all_features()
    
    # Get availability report
    # availability_report = engineer.get_feature_availability_report()
    # print(availability_report)
    
    # Get only high-quality features
    # high_avail_features = engineer.get_high_availability_features(threshold=0.7)
    # print(f"High availability features: {high_avail_features}")
    
    # Save to CSV
    # features_df.to_csv('financial_features.csv', index=False)
    
    print("✓ Financial Feature Engineering Module Ready")
    print("=" * 60)
    print("Available methods:")
    engineer_methods = [m for m in dir(FinancialFeatureEngineer) 
                       if not m.startswith('_') and callable(getattr(FinancialFeatureEngineer, m))]
    for method in sorted(engineer_methods):
        print(f"  • {method}()")

✓ Financial Feature Engineering Module Ready
Available methods:
  • asset_turnover()
  • cash_ratio()
  • cost_of_revenue_ratio()
  • create_all_features()
  • current_ratio()
  • days_inventory_outstanding()
  • days_sales_outstanding()
  • debt_composition()
  • debt_service_coverage()
  • debt_to_assets()
  • debt_to_equity()
  • depreciation_to_ppe()
  • ebit_to_revenue()
  • ebitda_margin()
  • equity_ratio()
  • get_feature_availability_report()
  • get_high_availability_features()
  • goodwill_ratio()
  • gross_profit_margin()
  • intangible_asset_ratio()
  • interest_coverage()
  • inventory_turnover()
  • long_term_debt_ratio()
  • net_profit_margin()
  • operating_expense_ratio()
  • operating_profit_margin()
  • other_raw_features()
  • quality_of_earnings()
  • quick_ratio()
  • rd_intensity()
  • receivables_turnover()
  • roa()
  • roe()
  • sga_ratio()
  • short_term_debt_ratio()
  • tangible_asset_ratio()
  • tax_rate()
  • working_capital_ratio()


In [22]:

if __name__ == "__main__":
    # Example: Load your data
    # df = pd.read_csv('your_financial_data.csv')
    
    # Create feature engineer
    # engineer = FinancialFeatureEngineer(df, min_data_availability=0.3)
    
    # Generate all features
    # features_df = engineer.create_all_features()
    
    # Get availability report
    # availability_report = engineer.get_feature_availability_report()
    # print(availability_report)
    
    # Get only high-quality features
    # high_avail_features = engineer.get_high_availability_features(threshold=0.7)
    # print(f"High availability features: {high_avail_features}")
    
    # Save to CSV
    # features_df.to_csv('financial_features.csv', index=False)
    
    print("✓ Financial Feature Engineering Module Ready")
    print("=" * 60)
    print("Available methods:")
    engineer_methods = [m for m in dir(FinancialFeatureEngineer) 
                       if not m.startswith('_') and callable(getattr(FinancialFeatureEngineer, m))]
    for method in sorted(engineer_methods):
        print(f"  • {method}()")

✓ Financial Feature Engineering Module Ready
Available methods:
  • asset_turnover()
  • cash_ratio()
  • cost_of_revenue_ratio()
  • create_all_features()
  • current_ratio()
  • days_inventory_outstanding()
  • days_sales_outstanding()
  • debt_composition()
  • debt_service_coverage()
  • debt_to_assets()
  • debt_to_equity()
  • depreciation_to_ppe()
  • ebit_to_revenue()
  • ebitda_margin()
  • equity_ratio()
  • get_feature_availability_report()
  • get_high_availability_features()
  • goodwill_ratio()
  • gross_profit_margin()
  • intangible_asset_ratio()
  • interest_coverage()
  • inventory_turnover()
  • long_term_debt_ratio()
  • net_profit_margin()
  • operating_expense_ratio()
  • operating_profit_margin()
  • other_raw_features()
  • quality_of_earnings()
  • quick_ratio()
  • rd_intensity()
  • receivables_turnover()
  • roa()
  • roe()
  • sga_ratio()
  • short_term_debt_ratio()
  • tangible_asset_ratio()
  • tax_rate()
  • working_capital_ratio()


In [23]:

df = pd.read_csv('data_clean.csv')
engineer = FinancialFeatureEngineer(df, min_data_availability=0.3)

features_df = engineer.create_all_features()


In [30]:

availability_report = engineer.get_feature_availability_report()
print(availability_report)


                       feature  availability_%
0                current_ratio           100.0
17  days_inventory_outstanding           100.0
31       short_term_debt_ratio           100.0
30        long_term_debt_ratio           100.0
29        tangible_asset_ratio           100.0
28              goodwill_ratio           100.0
27      intangible_asset_ratio           100.0
26             ebit_to_revenue           100.0
25                    tax_rate           100.0
24       cost_of_revenue_ratio           100.0
23         depreciation_to_ppe           100.0
22         quality_of_earnings           100.0
21                rd_intensity           100.0
20                   sga_ratio           100.0
19     operating_expense_ratio           100.0
18      days_sales_outstanding           100.0
16          inventory_turnover           100.0
1                  quick_ratio           100.0
15        receivables_turnover           100.0
14              asset_turnover           100.0
13           

In [32]:
# Select only missing columns
missing_cols = [col for col in df.columns if col.endswith("_missing")]

missing_flags_df = df[["ticker", "fiscalDateEnding"] + missing_cols]

# Merge
final_features = features_df.merge(missing_flags_df,
                                    on=["ticker", "fiscalDateEnding"],
                                    how="left")

In [33]:

high_avail_features = engineer.get_high_availability_features(threshold=0.7)
print(f"High availability features: {high_avail_features}")


High availability features: ['current_ratio', 'quick_ratio', 'cash_ratio', 'working_capital_ratio', 'gross_profit_margin', 'operating_profit_margin', 'net_profit_margin', 'ebitda_margin', 'roa', 'roe', 'debt_to_equity', 'debt_to_assets', 'equity_ratio', 'interest_coverage', 'asset_turnover', 'receivables_turnover', 'inventory_turnover', 'days_inventory_outstanding', 'days_sales_outstanding', 'operating_expense_ratio', 'sga_ratio', 'rd_intensity', 'quality_of_earnings', 'depreciation_to_ppe', 'cost_of_revenue_ratio', 'tax_rate', 'ebit_to_revenue', 'intangible_asset_ratio', 'goodwill_ratio', 'tangible_asset_ratio', 'long_term_debt_ratio', 'short_term_debt_ratio', 'debt_composition']


In [34]:

final_features.to_csv('financial_features.csv', index=False)


In [40]:
final_features.drop(['Unnamed: 0_was_missing'], axis=1, inplace=True)

In [41]:
final_features.columns

Index(['ticker', 'fiscalDateEnding', 'current_ratio', 'quick_ratio',
       'cash_ratio', 'working_capital_ratio', 'gross_profit_margin',
       'operating_profit_margin', 'net_profit_margin', 'ebitda_margin', 'roa',
       'roe', 'debt_to_equity', 'debt_to_assets', 'equity_ratio',
       'interest_coverage', 'debt_service_coverage', 'asset_turnover',
       'receivables_turnover', 'inventory_turnover',
       'days_inventory_outstanding', 'days_sales_outstanding',
       'operating_expense_ratio', 'sga_ratio', 'rd_intensity',
       'quality_of_earnings', 'depreciation_to_ppe', 'cost_of_revenue_ratio',
       'tax_rate', 'ebit_to_revenue', 'intangible_asset_ratio',
       'goodwill_ratio', 'tangible_asset_ratio', 'long_term_debt_ratio',
       'short_term_debt_ratio', 'debt_composition', 'log_assets',
       'share_growth', 'totalAssets_was_missing',
       'totalCurrentAssets_was_missing',
       'cashAndCashEquivalentsAtCarryingValue_was_missing',
       'cashAndShortTermInvestments

In [37]:

print("✓ Financial Feature Engineering Module Ready")
print("=" * 60)
print("Available methods:")
engineer_methods = [m for m in dir(FinancialFeatureEngineer) 
                   if not m.startswith('_') and callable(getattr(FinancialFeatureEngineer, m))]
for method in sorted(engineer_methods):
    print(f"  • {method}()")
    

✓ Financial Feature Engineering Module Ready
Available methods:
  • asset_turnover()
  • cash_ratio()
  • cost_of_revenue_ratio()
  • create_all_features()
  • current_ratio()
  • days_inventory_outstanding()
  • days_sales_outstanding()
  • debt_composition()
  • debt_service_coverage()
  • debt_to_assets()
  • debt_to_equity()
  • depreciation_to_ppe()
  • ebit_to_revenue()
  • ebitda_margin()
  • equity_ratio()
  • get_feature_availability_report()
  • get_high_availability_features()
  • goodwill_ratio()
  • gross_profit_margin()
  • intangible_asset_ratio()
  • interest_coverage()
  • inventory_turnover()
  • long_term_debt_ratio()
  • net_profit_margin()
  • operating_expense_ratio()
  • operating_profit_margin()
  • other_raw_features()
  • quality_of_earnings()
  • quick_ratio()
  • rd_intensity()
  • receivables_turnover()
  • roa()
  • roe()
  • sga_ratio()
  • short_term_debt_ratio()
  • tangible_asset_ratio()
  • tax_rate()
  • working_capital_ratio()


In [42]:
print("Shape:", final_features.shape)

Shape: (1360, 83)


In [44]:
print("Total NaNs:", final_features.isna().sum().sum())

Total NaNs: 1424


In [50]:
final_features.isna().sum().sort_values(ascending=False).head(10)

ticker                                    0
retainedEarnings_was_missing              0
otherNonCurrentLiabilities_was_missing    0
otherCurrentLiabilities_was_missing       0
shortLongTermDebtTotal_was_missing        0
longTermDebt_was_missing                  0
capitalLeaseObligations_was_missing       0
totalNonCurrentLiabilities_was_missing    0
shortTermDebt_was_missing                 0
currentAccountsPayable_was_missing        0
dtype: int64

In [47]:
final_features.drop(columns=["debt_service_coverage"], inplace=True)

In [48]:
final_features["share_growth"] = \
    final_features["share_growth"].fillna(0)

In [49]:
import numpy as np

final_features["interest_coverage"] = \
    final_features["interest_coverage"].replace([np.inf, -np.inf], np.nan)

final_features["interest_coverage"].fillna(0, inplace=True)

In [51]:
final_features.columns

Index(['ticker', 'fiscalDateEnding', 'current_ratio', 'quick_ratio',
       'cash_ratio', 'working_capital_ratio', 'gross_profit_margin',
       'operating_profit_margin', 'net_profit_margin', 'ebitda_margin', 'roa',
       'roe', 'debt_to_equity', 'debt_to_assets', 'equity_ratio',
       'interest_coverage', 'asset_turnover', 'receivables_turnover',
       'inventory_turnover', 'days_inventory_outstanding',
       'days_sales_outstanding', 'operating_expense_ratio', 'sga_ratio',
       'rd_intensity', 'quality_of_earnings', 'depreciation_to_ppe',
       'cost_of_revenue_ratio', 'tax_rate', 'ebit_to_revenue',
       'intangible_asset_ratio', 'goodwill_ratio', 'tangible_asset_ratio',
       'long_term_debt_ratio', 'short_term_debt_ratio', 'debt_composition',
       'log_assets', 'share_growth', 'totalAssets_was_missing',
       'totalCurrentAssets_was_missing',
       'cashAndCashEquivalentsAtCarryingValue_was_missing',
       'cashAndShortTermInvestments_was_missing', 'inventory_was_mi

In [52]:
final_features.to_csv('financial_features.csv', index=False)